# Fast-search mining: matched-question comparison

Compares greedy, beam-width 2, and beam-width 3 on the **same questions**. The analysis uses the exact three-way intersection of replay `example_id`s (expected: 165), never a positional slice. It also joins each replay to its mined trace when available to report oracle cross-entropy (CE) reductions.

Run this notebook from `analysis/`. It expects the files in `../outputs/fast_search_comparison/m3cot/`, as in the attached screenshot.

In [ ]:
from __future__ import annotations

import json
import re
from collections import Counter
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 100)

DATA_DIR = Path('../outputs/fast_search_comparison/m3cot')
EXPECTED_RUNS = {
    'Greedy': 'greedy',
    'Beam w=2': 'beam_search_width_2',
    'Beam w=3': 'beam_search_width_3',
}
assert DATA_DIR.exists(), f'Missing data directory: {DATA_DIR.resolve()}'
print(f'Data directory: {DATA_DIR.resolve()}')

In [ ]:
def read_jsonl(path: Path) -> list[dict[str, Any]]:
    with path.open(encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

def find_one(kind: str, token: str) -> Path | None:
    # A replay file is distinct from entropy tracking and summary files.
    candidates = [
        p for p in DATA_DIR.glob('*.jsonl')
        if token in p.name and (('_replay' in p.stem) if kind == 'replay' else ('traces' in p.stem))
        and 'entropy_tracking' not in p.name
    ]
    if len(candidates) != 1:
        print(f'{kind} candidates for {token}: {[p.name for p in candidates]}')
        return None
    return candidates[0]

def infer_run_from_name(path: Path) -> str | None:
    name = path.name
    for label, token in EXPECTED_RUNS.items():
        if token in name:
            return label
    return None

replay_paths = {label: find_one('replay', token) for label, token in EXPECTED_RUNS.items()}
trace_paths = {label: find_one('trace', token) for label, token in EXPECTED_RUNS.items()}
# In the supplied file listing the width-3 mining file has the legacy,
# unqualified name `...beam_search_fast...traces.jsonl`; use it only as this fallback.
if trace_paths['Beam w=3'] is None:
    legacy_w3 = [p for p in DATA_DIR.glob('*beam_search_fast*traces.jsonl') if 'entropy_tracking' not in p.name]
    if len(legacy_w3) == 1:
        trace_paths['Beam w=3'] = legacy_w3[0]
        print(f"Using legacy width-3 trace filename: {legacy_w3[0].name}")
missing_replays = [label for label, path in replay_paths.items() if path is None]
if missing_replays:
    raise FileNotFoundError(f'Missing or ambiguous replay files for: {missing_replays}. Check DATA_DIR and filename tokens.')

display(pd.DataFrame({
    'run': EXPECTED_RUNS,
    'replay_file': {k: v.name if v else None for k, v in replay_paths.items()},
    'trace_file': {k: v.name if v else None for k, v in trace_paths.items()},
}).reset_index(drop=True))

In [ ]:
def numeric(value: Any) -> float:
    converted = pd.to_numeric(value, errors='coerce')
    return float(converted) if pd.notna(converted) else np.nan

def action_counts(trace: Any) -> Counter:
    # Replay rows use trace entries with `action`; mined rows use `type`.
    return Counter(str(a.get('action', a.get('type', ''))).upper() for a in (trace or []))

def replay_frame(label: str, path: Path) -> pd.DataFrame:
    records = []
    for row in read_jsonl(path):
        entropy = row.get('answer_option_entropy') or {}
        counts = action_counts(row.get('trace'))
        records.append({
            'run': label, 'example_id': str(row['example_id']), 'question': row.get('question'),
            'correct': bool(row.get('correct')), 'decoded_answer': row.get('decoded_answer'),
            'gold_answer': row.get('gold_answer'), 'num_trace_actions': numeric(row.get('num_trace_actions')),
            'num_output_tokens': numeric(row.get('num_output_tokens')),
            'answer_option_entropy': numeric(entropy.get('entropy', row.get('answer_option_entropy'))),
            'decoded_token_entropy_mean': numeric(row.get('decoded_token_entropy_mean')),
            'trace_attention_mass': numeric(row.get('trace_attention_mass')),
            'visual_trace_attention_mass': numeric(row.get('visual_trace_attention_mass')),
            'think_attention_mass': numeric(row.get('think_attention_mass')),
            'patch_actions': counts['PATCH'], 'think_actions': counts['THINK'],
            'stop_actions': counts['STOP'],
        })
    return pd.DataFrame(records).drop_duplicates('example_id', keep='last')

replays = {label: replay_frame(label, path) for label, path in replay_paths.items()}
id_sets = {label: set(frame.example_id) for label, frame in replays.items()}
shared_ids = set.intersection(*id_sets.values())
min_count = min(map(len, id_sets.values()))
print('Replay rows:', {label: len(frame) for label, frame in replays.items()})
print(f'Three-way intersection: {len(shared_ids)} IDs; minimum completed run: {min_count}')
if len(shared_ids) != min_count:
    print('WARNING: the short run is not a subset of both longer runs. Results use only the true ID intersection.')

matched = pd.concat([frame[frame.example_id.isin(shared_ids)] for frame in replays.values()], ignore_index=True)
assert matched.groupby('run').example_id.nunique().eq(len(shared_ids)).all()
display(matched.groupby('run').agg(questions=('example_id', 'nunique'), correct=('correct', 'sum')).assign(accuracy=lambda x: x.correct / x.questions))

In [ ]:
def selected_trajectory(row: dict[str, Any]) -> dict[str, Any]:
    trajectories = row.get('beam_trajectories') or []
    if trajectories:
        return min(trajectories, key=lambda t: (int(t.get('rank', 10**9)), float(t.get('weighted_ce', np.inf))))
    return row

def trace_ce_frame(label: str, path: Path | None) -> pd.DataFrame:
    if path is None:
        return pd.DataFrame(columns=['run', 'example_id'])
    records = []
    for row in read_jsonl(path):
        traj = selected_trajectory(row)
        decisions = traj.get('decisions') or row.get('decisions') or []
        before = next((d.get('ce_before') for d in decisions if d.get('ce_before') is not None), np.nan)
        final = traj.get('weighted_ce', next((d.get('weighted_ce') for d in reversed(decisions) if d.get('weighted_ce') is not None), np.nan))
        # Each mining stage has a different remaining-rationale target, so a
        # first-stage CE minus final-stage CE is not meaningful. Sum only the
        # within-stage CE improvements, where both scores share a target.
        stage_improvements = [numeric(d.get('improvement', numeric(d.get('ce_before')) - numeric(d.get('ce_selected')))) for d in decisions]
        stage_relative_improvements = [100 * imp / numeric(d.get('ce_before')) for imp, d in zip(stage_improvements, decisions) if pd.notna(imp) and pd.notna(numeric(d.get('ce_before'))) and numeric(d.get('ce_before')) != 0]
        rationale_final = traj.get('ce_rationale', next((d.get('ce_rationale') for d in reversed(decisions) if d.get('ce_rationale') is not None), np.nan))
        answer_final = traj.get('ce_answer', next((d.get('ce_answer') for d in reversed(decisions) if d.get('ce_answer') is not None), np.nan))
        counts = action_counts(traj.get('trace') or row.get('trace'))
        records.append({
            'run': label, 'example_id': str(row['example_id']), 'mining_ce_before': numeric(before),
            'mining_weighted_ce_final': numeric(final), 'mining_ce_reduction': np.nansum(stage_improvements),
            'mining_ce_reduction_pct': np.nanmean(stage_relative_improvements) if stage_relative_improvements else np.nan,
            'mining_rationale_ce_final': numeric(rationale_final), 'mining_answer_ce_final': numeric(answer_final),
            'mined_patch_actions': counts['PATCH'], 'mined_think_actions': counts['THINK'],
        })
    return pd.DataFrame(records).drop_duplicates('example_id', keep='last')

trace_metrics = pd.concat([trace_ce_frame(label, path) for label, path in trace_paths.items()], ignore_index=True)
if trace_metrics.empty:
    print('Trace files were not found. Replay metrics will still be reported; CE-reduction columns will be unavailable.')
else:
    matched = matched.merge(trace_metrics, on=['run', 'example_id'], how='left', validate='one_to_one')
    coverage = matched.groupby('run').mining_ce_reduction.notna().mean()
    display(coverage.rename('trace CE coverage').to_frame())

In [ ]:
metric_columns = [
    'correct', 'num_trace_actions', 'num_output_tokens', 'patch_actions', 'think_actions',
    'answer_option_entropy', 'decoded_token_entropy_mean', 'trace_attention_mass',
    'visual_trace_attention_mass', 'think_attention_mass', 'mining_ce_before',
    'mining_weighted_ce_final', 'mining_ce_reduction', 'mining_ce_reduction_pct',
    'mining_rationale_ce_final', 'mining_answer_ce_final', 'mined_patch_actions', 'mined_think_actions',
]
available = [c for c in metric_columns if c in matched]
summary = matched.groupby('run')[available].agg(['count', 'mean', 'median', 'std']).T
summary.index = [f'{metric} — {stat}' for metric, stat in summary.index]
display(summary)

# Accuracy transitions reveal which questions a method gains or loses, beyond aggregate accuracy.
correct_matrix = matched.pivot(index='example_id', columns='run', values='correct').reindex(columns=EXPECTED_RUNS)
patterns = correct_matrix.value_counts().rename('questions').reset_index()
patterns['pattern'] = patterns[list(EXPECTED_RUNS)].astype(int).astype(str).agg(' / '.join, axis=1)
display(patterns[['pattern', 'questions']].sort_values('questions', ascending=False))

question_lookup = matched.drop_duplicates('example_id').set_index('example_id').question
disagreements = correct_matrix[correct_matrix.nunique(axis=1) > 1].copy()
disagreements.insert(0, 'question', disagreements.index.map(question_lookup))
display(disagreements.sort_index())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10), constrained_layout=True)
order = list(EXPECTED_RUNS)
accuracy = matched.groupby('run').correct.mean().reindex(order)
axes[0, 0].bar(order, accuracy, color=sns.color_palette('deep', 3))
axes[0, 0].set(title=f'Accuracy on the shared {len(shared_ids)} questions', ylabel='Accuracy', ylim=(0, 1))
for i, value in enumerate(accuracy): axes[0, 0].text(i, value + .02, f'{value:.1%}', ha='center')

sns.boxplot(data=matched, x='run', y='num_trace_actions', order=order, ax=axes[0, 1])
axes[0, 1].set(title='Replay trace length', xlabel='', ylabel='Actions')

if 'mining_ce_reduction' in matched and matched.mining_ce_reduction.notna().any():
    sns.boxplot(data=matched, x='run', y='mining_ce_reduction', order=order, ax=axes[1, 0])
    axes[1, 0].axhline(0, color='black', linewidth=1)
    axes[1, 0].set(title='Cumulative stage-local oracle CE reduction', xlabel='', ylabel='Σ (CE before − CE selected)')
else:
    axes[1, 0].text(.5, .5, 'Trace files unavailable', ha='center', va='center'); axes[1, 0].set_axis_off()

entropy_col = 'answer_option_entropy' if matched.answer_option_entropy.notna().any() else 'decoded_token_entropy_mean'
sns.boxplot(data=matched, x='run', y=entropy_col, order=order, ax=axes[1, 1])
axes[1, 1].set(title=f"Replay {entropy_col.replace('_', ' ')}", xlabel='', ylabel='Entropy')
plt.show()

# Optional export for paper tables / further analysis.
out = Path('fast_search_intersection_matched_metrics.csv')
matched.sort_values(['example_id', 'run']).to_csv(out, index=False)
print(f'Wrote matched per-question metrics to {out.resolve()}')